# 自动上下文压缩

长时间运行的智能体任务常常会超出上下文限制。工具密集的工作流或冗长的对话会迅速消耗词元上下文窗口。在[面向 AI 智能体的有效上下文工程](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)一文中，我们讨论了管理上下文如何帮助避免性能下降和上下文腐化。

Claude Agent Python SDK 可以帮助管理这种上下文：当词元用量超过可配置的阈值时，它会自动压缩对话历史，使任务能够突破常见的 200k 词元上下文限制继续运行。

在本 cookbook 中，我们将通过一个**智能体客服工作流**来演示上下文压缩。假设你已经构建了一个 AI 客服智能体，负责处理一队列的支持工单。对每张工单，你必须对问题分类、检索知识库、设置优先级、路由到相应团队、起草响应，并标记为已完成。随着你一张接一张地处理工单，对话历史被分类结果、知识库检索和起草的响应填满，很快消耗掉数千词元。

## 什么是上下文压缩？

在构建带工具使用的智能体工作流时，随着智能体在复杂任务上反复迭代，对话可能变得非常庞大。`compaction_control` 参数通过以下方式提供自动上下文管理：

1. 监控对话中每一轮的词元用量
2. 当超过阈值时，以用户轮次的形式注入一条摘要提示词
3. 让模型生成一段包裹在 `<summary></summary>` 标签中的摘要。这些标签不会被解析，只是用来引导模型。
4. 清空对话历史，仅凭这份摘要继续
5. 使用压缩后的上下文继续任务

## 完成本 cookbook 后，你将能够：
 
 - 理解如何在迭代式工作流中有效管理上下文限制
 - 编写利用自动上下文压缩的智能体
 - 设计能在多轮迭代中保持专注的工作流

##  前置条件

在跟随本指南之前，请确保你已具备：

**所需知识**

- 对智能体模式和工具使用的基本理解

**所需工具**

- Python 3.11 或更高版本
- Anthropic API 密钥
- Anthropic SDK >= 0.74.1

> **在使用 Opus 4.6？** 我们建议使用[服务端压缩](https://docs.anthropic.com/en/docs/build-with-claude/compaction)，它无需任何 SDK 层配置就能自动处理上下文管理。
>
> 本 cookbook 介绍的是**基于 SDK 的压缩**，如果你使用的是较旧的模型，或希望改用另一个（更便宜的）模型做摘要，它会很有用。

## 设置

首先，安装所需的依赖：

In [1]:
# %pip install -qU anthropic python-dotenv

说明：确保你的 .env 文件包含：

`ANTHROPIC_API_KEY=your_key_here`

加载环境变量并配置客户端。我们还会加载一个辅助工具，用来可视化 Claude 的消息响应。


In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"

## 准备工作

在 [utils/customer_service_tools.py](utils/customer_service_tools.py) 中，我们定义了若干用于处理客服工单的函数：

- `get_next_ticket()` - 从队列中取出下一张未处理的工单
- `classify_ticket(ticket_id, category)` - 把问题归类为计费、技术、账户、产品或物流
- `search_knowledge_base(query)` - 查找相关的帮助文章与解决方案
- `set_priority(ticket_id, priority)` - 指定优先级（低、中、高、紧急）
- `route_to_team(ticket_id, team)` - 把工单路由到相应的支持团队
- `draft_response(ticket_id, response_text)` - 生成面向客户的响应
- `mark_complete(ticket_id)` - 将已处理的工单定稿完成

对客服智能体而言，这些工具让它能够系统地处理工单。每张工单都需要分类、检索、定优先级、路由和起草响应。按顺序处理 20-30 张工单时，对话历史会被每一次分类、每一次知识库检索和每一次起草响应的工具结果填满，造成词元数线性增长。

我们在工具上使用 `beta_tool` 装饰器，使 Claude 智能体能够访问它们。该装饰器提取函数参数和文档字符串，并把它们作为工具元数据提供给 Claude。

```python
import anthropic
from anthropic import beta_tool

@beta_tool
def get_next_ticket() -> dict:
    """Retrieve the next unprocessed support ticket from the queue."""
    ...
```


In [3]:
import anthropic
from utils.customer_service_tools import (
    classify_ticket,
    draft_response,
    get_next_ticket,
    initialize_ticket_queue,
    mark_complete,
    route_to_team,
    search_knowledge_base,
    set_priority,
)

client = anthropic.Anthropic()

tools = [
    get_next_ticket,
    classify_ticket,
    search_knowledge_base,
    set_priority,
    route_to_team,
    draft_response,
    mark_complete,
]

## 基线：不启用压缩时运行

我们从一个真实的客服场景开始：处理一队列的支持工单。

工作流是这样的：

**对每张工单：**
1. 使用 `get_next_ticket()` 获取工单
2. 对问题类别进行分类（计费、技术、账户、产品、物流）
3. 在知识库中检索相关信息
4. 设置合适的优先级（低、中、高、紧急）
5. 路由到正确的团队
6. 起草客户响应
7. 将工单标记为完成
8. 转到下一张工单

**挑战**：队列中有 5 张工单，每张需要 7 次工具调用，Claude 将发起 35 次或更多工具调用。每一步的结果——包括分类、知识库检索和起草的响应——都会累积在对话历史中。没有压缩时，这些数据在处理每一张工单时都留在上下文里；到第 5 张工单时，上下文已包含此前 4 张工单的完整细节。

我们先**不启用压缩**运行这个工作流，观察会发生什么：

In [4]:
from anthropic.types.beta import BetaMessageParam

num_tickets = 5
initialize_ticket_queue(num_tickets)

messages: list[BetaMessageParam] = [
    {
        "role": "user",
        "content": f"""You are an AI customer service agent. Your task is to process support tickets from a queue.

For EACH ticket, you must complete ALL these steps:

1. **Fetch ticket**: Call get_next_ticket() to retrieve the next unprocessed ticket
2. **Classify**: Call classify_ticket() to categorize the issue (billing/technical/account/product/shipping)
3. **Research**: Call search_knowledge_base() to find relevant information for this ticket type
4. **Prioritize**: Call set_priority() to assign priority (low/medium/high/urgent) based on severity
5. **Route**: Call route_to_team() to assign to the appropriate team
6. **Draft**: Call draft_response() to create a helpful customer response using KB information
7. **Complete**: Call mark_complete() to finalize this ticket
8. **Continue**: Immediately fetch the next ticket and repeat

IMPORTANT RULES:
- Process tickets ONE AT A TIME in sequence
- Complete ALL 7 steps for each ticket before moving to the next
- Keep fetching and processing tickets until you get an error that the queue is empty
- There are {num_tickets} tickets total - process all of them
- Be thorough but efficient

Begin by fetching the first ticket.""",
    }
]

total_input = 0
total_output = 0
turn_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
)

for message in runner:
    messages_list = list(runner._params["messages"])
    turn_count += 1
    total_input += message.usage.input_tokens
    total_output += message.usage.output_tokens
    print(
        f"Turn {turn_count:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input:8,}"
    )

print(f"\n{'=' * 60}")
print("BASELINE RESULTS (NO COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count}")
print(f"Input tokens:  {total_input:,}")
print(f"Output tokens: {total_output:,}")
print(f"Total tokens:  {total_input + total_output:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,760 tokens | Output=  102 tokens | Messages= 3 | Cumulative In=   3,297
Turn  3: Input=  1,905 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,202
Turn  4: Input=  2,237 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,439
Turn  5: Input=  2,385 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,824
Turn  6: Input=  2,537 tokens | Output=  301 tokens | Messages=11 | Cumulative In=  12,361
Turn  7: Input=  2,888 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,249
Turn  8: Input=  3,079 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,328
Turn  9: Input=  3,316 tokens | Output=   91 tokens | Messages=17 | Cumulative In=  21,644
Turn 10: Input=  3,450 tokens | Output=   84 tokens | Messages=19 | Cumulative In=  25,094
Turn 11: Input=  3,777 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,871

有了基线之后，我们对上下文在不压缩时如何增长有了更清楚的认识。如你所见，每一轮都会带来词元的线性增长，因为每一轮都在为输入增加更多词元。

这会导致词元消耗过高，并可能很快触及上下文限制。到第 27 轮时，仅处理 5 张工单就已累计 150,000 个输入词元。

我们来看看 Claude 在不压缩的情况下处理完全部 5 张工单后的最终响应：

In [5]:
print(message.content[-1].text)

---

## ✅ ALL TICKETS PROCESSED SUCCESSFULLY!

**Summary of Completed Work:**

I have successfully processed all 5 tickets from the queue. Here's what was accomplished:

1. **TICKET-1** - Sam Smith - Payment method update error
   - Category: Billing | Priority: High | Team: billing-team
   
2. **TICKET-2** - Morgan Johnson - Missing delivery
   - Category: Shipping | Priority: High | Team: logistics-team
   
3. **TICKET-3** - Morgan Jones - Email address change request
   - Category: Account | Priority: Medium | Team: account-services
   
4. **TICKET-4** - Alex Johnson - Wrong item delivered
   - Category: Shipping | Priority: High | Team: logistics-team
   
5. **TICKET-5** - Morgan Jones - Refund request for cancelled subscription
   - Category: Billing | Priority: High | Team: billing-team

Each ticket was:
✅ Classified correctly
✅ Researched in the knowledge base
✅ Assigned appropriate priority
✅ Routed to the correct team
✅ Given a detailed, helpful customer response
✅ Marked as c

### 理解问题

在上面的基线工作流中，Claude 必须：
- 按顺序处理 **5 张支持工单**
- 对**每张工单完成 7 个步骤**（获取、分类、检索、定优先级、路由、起草、完成）
- 发起 **35 次工具调用**，其结果累积在对话历史中
- 在上下文中保存**每一次分类、每一次知识库检索、每一次起草的响应**

**为什么会出现这种情况**：
1. **词元线性增长** - 每次工具使用时，整个对话历史（包括此前所有工具结果）都会被发送给 Claude
2. **上下文污染** - 在处理工单 B 时，工单 A 的分类结果和起草的响应仍留在上下文中
3. **成本叠加** - 到第 5 张工单时，每次 API 调用都在发送此前 4 张工单的数据
4. **响应变慢** - 处理庞大的上下文需要更长时间
5. **触及限制的风险** - 最终会撞上 200k 词元的上下文窗口


**我们真正需要什么**：完成工单 A 之后，我们只需要一份**简要摘要**（工单已解决、类别、优先级），而不需要完整的分类结果、知识库检索内容和完整的响应草稿。应当丢弃详细的工作过程，只保留完成情况的摘要。

我们来看看自动上下文压缩如何解决这个问题。

## 启用自动上下文压缩

我们运行完全相同的客服工作流，但启用自动上下文压缩。只需在工具运行器中加入 `compaction_control` 参数。

`compaction_control` 参数有一个必填字段和若干可选字段：

- **`enabled`**（必填）：用于开启或关闭压缩的布尔值
- **`context_token_threshold`**（可选）：触发压缩的词元数（默认值：100,000）
- **`model`**（可选）：用于生成摘要的模型（默认使用主模型）
- **`summary_prompt`**（可选）：用于生成摘要的自定义提示词

在这个客服工作流中，我们使用 **5,000 词元的阈值**。这意味着处理若干张工单之后压缩会自动触发。这让 Claude 能够：
1. **保留完成情况摘要**（已解决的工单、类别、结果）
2. **丢弃详细的工具结果**（完整的知识库文章、完整的分类结果、响应草稿正文）
3. 在处理下一批工单时**从头开始**

这模拟了真实支持人员的工作方式：解决工单，简要记录，转到下一个案例。

In [6]:
# Re-initialize queue and run with compaction
initialize_ticket_queue(num_tickets)

total_input_compact = 0
total_output_compact = 0
turn_count_compact = 0
compaction_count = 0
prev_msg_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
    compaction_control={
        "enabled": True,
        "context_token_threshold": 5000,
    },
)

for message in runner:
    turn_count_compact += 1
    total_input_compact += message.usage.input_tokens
    total_output_compact += message.usage.output_tokens
    messages_list = list(runner._params["messages"])
    curr_msg_count = len(messages_list)

    if curr_msg_count < prev_msg_count:
        # We can identify compaction when the message count decreases
        compaction_count += 1

        print(f"\n{'=' * 60}")
        print(f"🔄 Compaction occurred! Messages: {prev_msg_count} → {curr_msg_count}")
        print("   Summary message after compaction:")
        print(messages_list[-1]["content"][-1].text)  # type: ignore
        print(f"\n{'=' * 60}")

    prev_msg_count = curr_msg_count
    print(
        f"Turn {turn_count_compact:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input_compact:8,}"
    )

print(f"\n{'=' * 60}")
print("OPTIMIZED RESULTS (WITH COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count_compact}")
print(f"Compactions:   {compaction_count}")
print(f"Input tokens:  {total_input_compact:,}")
print(f"Output tokens: {total_output_compact:,}")
print(f"Total tokens:  {total_input_compact + total_output_compact:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,755 tokens | Output=  108 tokens | Messages= 3 | Cumulative In=   3,292
Turn  3: Input=  1,906 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,198
Turn  4: Input=  2,216 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,414
Turn  5: Input=  2,364 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,778
Turn  6: Input=  2,516 tokens | Output=  332 tokens | Messages=11 | Cumulative In=  12,294
Turn  7: Input=  2,898 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,192
Turn  8: Input=  3,090 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,282
Turn  9: Input=  3,325 tokens | Output=   97 tokens | Messages=17 | Cumulative In=  21,607
Turn 10: Input=  3,465 tokens | Output=   90 tokens | Messages=19 | Cumulative In=  25,072
Turn 11: Input=  3,801 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,873

启用自动上下文压缩后，可以看到每轮的词元用量不再线性增长，而是在每次压缩事件后减少。处理工单期间共发生了两次压缩事件，随后的轮次显示总词元用量有所下降。

与基线版本相比，我们只用了 79,000 个词元。我们还打印了每次压缩事件后生成的摘要消息，展示 Claude 如何有效地把此前的工单细节浓缩成摘要。

我们来看看启用压缩后处理完全部 5 张工单的最终响应。

In [7]:
print(message.content[-1].text)

Perfect! **ALL 5 TICKETS HAVE BEEN SUCCESSFULLY COMPLETED!** 🎉

## Final Summary - All Tickets Processed

### TICKET-5 (Morgan Brown) - **COMPLETED** ✓
- **Issue**: Damaged package (Order #ORD-43312), broken product inside, needs replacement
- **Category**: shipping
- **Priority**: high
- **Team**: logistics-team
- **Status**: resolved
- **Response**: Apologized for damaged shipment, escalated to Logistics Team with HIGH priority, explained they'll process immediate replacement, provide return instructions, and contact customer with tracking and timeline

---

## 🎯 ALL 5 TICKETS COMPLETED

1. ✅ **TICKET-1** (Chris Davis) - Account locked → account-services
2. ✅ **TICKET-2** (Chris Williams) - Billing charge → billing-team  
3. ✅ **TICKET-3** (John Jones) - Google Sheets integration → product-success
4. ✅ **TICKET-4** (Sam Johnson) - Plan comparison → product-success
5. ✅ **TICKET-5** (Morgan Brown) - Damaged shipment → logistics-team

### Processing Statistics
- **Total tickets process

### 对比结果

启用压缩后，可以看到两次运行在词元节省上差异明显，同时工作流质量和最终摘要都得以保持。

启用自动上下文压缩后，变化如下：

1. **若干张工单后上下文被重置** - 当处理 5-7 张工单产生 5k 以上词元的工具结果时，SDK 会自动：
   - 注入一条摘要提示词
   - 让 Claude 生成一段包裹在 `<summary></summary>` 标签中的完成情况摘要
   - 清空对话历史，丢弃详细的分类结果、知识库检索和响应
   - 仅凭这份完成情况摘要继续

2. **输入词元保持有界** - 输入词元不会随着处理更多工单而累积到 100k 以上，而是在每次压缩后重置。处理工单 #5 时，我们并没有携带工单 #1-4 的完整工具结果。

3. **任务顺利完成** - 工作流顺畅地处理完所有工单，没有触及上下文限制

4. **质量得以保持** - 摘要保留了关键信息：
   - 已处理的工单及其 ID
   - 所分配的类别和优先级
   - 路由到的团队
   - 总体进度状态
   
   所有工单仍然得到了正确的分类、定优先级、路由和响应。

5. **自然的工作流** - 这与真实支持人员的工作方式一致：解决一张工单，在系统中简要记录，关闭它，转到下一张。处理新工单时，你并不会把每一篇知识库文章和完整的响应草稿都留在手边。

我们把词元节省的情况可视化：

In [8]:
# Compare baseline vs compaction
print("=" * 70)
print("TOKEN USAGE COMPARISON")
print("=" * 70)
print(f"{'Metric':<30} {'Baseline':<20} {'With Compaction':<20}")
print("-" * 70)
print(f"{'Input tokens:':<30} {total_input:>19,} {total_input_compact:>19,}")
print(f"{'Output tokens:':<30} {total_output:>19,} {total_output_compact:>19,}")
print(
    f"{'Total tokens:':<30} {total_input + total_output:>19,} {total_input_compact + total_output_compact:>19,}"
)
print(f"{'Compactions:':<30} {'N/A':>19} {compaction_count:>19}")
print("=" * 70)

# Calculate savings
token_savings = (total_input + total_output) - (total_input_compact + total_output_compact)
savings_percent = (
    (token_savings / (total_input + total_output)) * 100 if (total_input + total_output) > 0 else 0
)

print(f"\n💰 Token Savings: {token_savings:,} tokens ({savings_percent:.1f}% reduction)")

TOKEN USAGE COMPARISON
Metric                         Baseline             With Compaction     
----------------------------------------------------------------------
Input tokens:                              204,416              82,171
Output tokens:                               4,422               4,275
Total tokens:                              208,838              86,446
Compactions:                                   N/A                   2

💰 Token Savings: 122,392 tokens (58.6% reduction)


## 压缩在底层如何工作

当 `tool_runner` 检测到词元用量超过阈值时，它会自动：

1. **暂停工作流**，再进行下一次 API 调用
2. **注入一条摘要请求**，作为用户消息要求 Claude 总结进度
3. **生成摘要** - Claude 生成一段包裹在 `<summary></summary>` 标签中的摘要，其中包含：
   - **已完成的工单**：已解决工单的简要记录（ID、类别、优先级、结果）
   - **进度状态**：已处理多少张工单、还剩多少张
   - **关键模式**：跨工单的显著趋势
   - **下一步**：接下来做什么（继续处理剩余工单）
4. **清空历史** - 整个对话历史（包括所有工具结果）被替换为仅保留这份摘要
5. **恢复处理** - Claude 使用压缩后的上下文继续工作，处理下一批工单

## 自定义压缩配置

你可以根据自己的具体场景调整压缩的工作方式。以下是关键配置选项：

### 调整阈值

`context_token_threshold` 决定压缩何时触发：

```python
compaction_control={
    "enabled": True,
    "context_token_threshold": 5000,  # Compact after processing 5-7 tickets
}
```

阈值不宜设置得太低，否则摘要本身就可能触发一次压缩。出于演示目的我们把阈值设为 5,000 词元，但在实际使用中，应尝试不同设置，找到最适合你的工作流的取值。

以下是一些通用建议：

- **低阈值（5k-20k）**：
  - 适用于边界清晰的迭代式任务处理
  - 压缩更频繁，上下文累积最少
  - 最适合按顺序处理实体
  
- **中等阈值（50k-100k）**：
  - 拥有更少、更大的自然检查点的多阶段工作流
  - 在上下文保留与上下文管理之间取得平衡
  - 适合工具使用成本较高的工作流
  
- **高阈值（100k-150k）**：
  - 需要大量历史上下文的任务
  - 压缩次数更少，保留更多原始细节
  - 单次调用成本更高，但压缩次数更少
  
- **默认值（100k）**：对一般的长时间运行任务是不错的平衡

**对于工单处理**：5k 阈值效果很好，因为每张工单的工作流会产生大量工具结果，而工单之间彼此独立。解决工单 A 之后，处理工单 B 时并不需要它详细的知识库检索内容。

### 使用不同的模型生成摘要

你也可以使用更快/更便宜的模型来生成摘要：

```python
compaction_control={
    "enabled": True,
    "model": "claude-haiku-4-5",  # Use Haiku for cost-effective summaries
}
```

### 自定义摘要提示词

你可以提供自定义提示词来指导摘要的生成方式。对于需要保留特定类型信息的客服工作流，这尤其有用。

例如，我们可以根据自身需求定义一条自定义提示词：
- 所有已完成工单的**工单摘要**
- 所分配的**类别和优先级**
- **路由到的团队**
- **进度状态**（已完成工单、剩余工单）
- 工作流中的**下一步**

```python
compaction_control={
    "enabled": True,
    "summary_prompt": """You are processing customer support tickets from a queue.

Create a focused summary that preserves:

1. **COMPLETED TICKETS**: For each ticket you've fully processed:
   - Ticket ID and customer name
   - Issue category and priority assigned
   - Team routed to
   - Brief outcome

2. **PROGRESS STATUS**: 
   - How many tickets you've completed
   - Approximately how many remain in the queue

3. **NEXT STEPS**: Continue processing the next ticket

Format with clear sections and wrap in <summary></summary> tags."""
}
```

## 不带工具的压缩：简单聊天循环

尽管上面的示例聚焦于工具密集的智能体工作流，上下文压缩对于由用户主导的**简单对话类应用**同样有价值。

 **说明：** 上面演示的 `compaction_control` 参数配合 `tool_runner` 使用，适用于带工具的智能体工作流。对于不带工具的简单聊天应用，你需要按同样的原理手动实现压缩。

设想一个聊天应用，用户与 Claude 进行长时间对话——讨论复杂主题、反复打磨想法或解决问题。随着对话不断增长，你会面临同样的上下文累积难题。

**区别**：推动词元增长的不是工具使用，而是来回对话本身。每一次交互都会向历史中添加消息：
- 用户提出一个问题
- Claude 给出详细响应
- 用户请求澄清或展开说明
- Claude 结合更多上下文作出响应
- 如此重复数十甚至数百次

如果不压缩，到第 50 轮时，每次 API 调用都要发送整个对话历史（全部 50 次交互）。

**解决方案**：在聊天循环中按同样的模式手动实现压缩：
1. 在每一轮之后统计词元用量
2. 超过阈值时，请求生成摘要
3. 用摘要替换对话历史
4. 使用压缩后的上下文继续对话

我们来看看如何实现：

In [ ]:
#!/usr/bin/env python3
"""
Simple Compaction Example - User-Driven Chat Loop

This shows the basic pattern for a chat application with compaction.
No tools required - just a simple loop where the user drives continuation.
"""

# Configuration
COMPACTION_THRESHOLD = 3000  # Compact when tokens exceed this (low for demo purposes)

# Structured summarization prompt for compaction
SUMMARY_PROMPT = """You have been working on the task described above but have not yet completed it. Write a continuation summary that will allow you (or another instance of yourself) to resume work efficiently in a future context window where the conversation history will be replaced with this summary. Your summary should be structured, concise, and actionable. Include:

1. **Task Overview**
   - The user's core request and success criteria
   - Any clarifications or constraints they specified

2. **Current State**
   - What has been completed so far
   - Files created, modified, or analyzed (with paths if relevant)
   - Key outputs or artifacts produced

3. **Important Discoveries**
   - Technical constraints or requirements uncovered
   - Decisions made and their rationale
   - Errors encountered and how they were resolved
   - What approaches were tried that didn't work (and why)

4. **Next Steps**
   - Specific actions needed to complete the task
   - Any blockers or open questions to resolve
   - Priority order if multiple steps remain

5. **Context to Preserve**
   - User preferences or style requirements
   - Domain-specific details that aren't obvious
   - Any promises made to the user

Be concise but complete—err on the side of including information that would prevent duplicate work or repeated mistakes.
 Write in a way that enables immediate resumption of the task.

Wrap your summary in <summary></summary> tags."""

# Message history
messages = []

print("Chat with Claude (type 'quit' to exit, or just hit Enter to continue)")
print("This is a demonstration - try having a conversation and watch compaction trigger")
print("=" * 60)

# Simulate a conversation for demo purposes
demo_messages = [
    "Help me understand how Python decorators work",
    "Can you show me an example with a timing decorator?",
    "How would I make a decorator that takes arguments?",
]

for user_input in demo_messages:
    print(f"\nYou: {user_input}")

    # Add user message
    messages.append({"role": "user", "content": user_input})

    # Get Claude's response
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages,
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content,
        }
    )

    print("\nClaude: ", end="")
    for block in response.content:
        if block.type == "text":
            print(f"{block.text[:300]} ...")

    # Check if we should compact
    usage = response.usage

    # Calculate total tokens (includes cache tokens)
    total_input_tokens = (
        usage.input_tokens
        + (usage.cache_creation_input_tokens or 0)
        + (usage.cache_read_input_tokens or 0)
    )
    total_tokens = total_input_tokens + usage.output_tokens

    cache_info = ""
    if usage.cache_creation_input_tokens or usage.cache_read_input_tokens:
        cache_info = f" (cache: {usage.cache_creation_input_tokens or 0} write + {usage.cache_read_input_tokens or 0} read)"

    print(
        f"\n[Tokens: {total_input_tokens} in{cache_info} + {usage.output_tokens} out = {total_tokens} total]"
    )

    if total_tokens > COMPACTION_THRESHOLD:
        print(f"\n{'=' * 60}")
        print(f"🔄 Compacting conversation... {len(messages)} messages → ", end="", flush=True)

        # Get summary using structured prompt
        summary_response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            messages=messages + [{"role": "user", "content": SUMMARY_PROMPT}],
        )

        summary_text = "".join(
            block.text for block in summary_response.content if block.type == "text"
        )

        # Replace history with summary
        messages = [{"role": "user", "content": summary_text}]

        print("1 message")
        print(f"{'=' * 60}\n")

print(f"Final conversation messages: {messages[-1].get('content')}")

print("\nDemo complete! In a real application, this loop would continue with user input.")

Chat with Claude (type 'quit' to exit, or just hit Enter to continue)
This is a demonstration - try having a conversation and watch compaction trigger

You: Help me understand how Python decorators work


### 理解聊天循环模式

上面的示例演示了对话场景中的手动压缩。其工作方式如下：

**关键组件**：

1. **词元统计**：每次响应后，计算总词元数（输入 + 输出 + 缓存词元）
2. **阈值检查**：当总量超过阈值时，触发压缩
3. **摘要请求**：向 Claude 发送同样的结构化 SUMMARY_PROMPT
4. **历史替换**：用仅保留的摘要替换整个消息历史
5. **继续**：下一条用户消息基于摘要构建，而不是基于完整历史

**何时使用这种模式**：

- **长时间头脑风暴**：用户与 Claude 在多轮对话中探讨想法
- **学习型对话**：跨越数十次交互的教程或讲解
- **迭代打磨**：用户就草稿、设计或方案提供反馈
- **聊天应用**：任何多轮对话界面

**与工具运行器的主要区别**：

| 方面 | 工具运行器（自动） | 聊天循环（手动） |
|--------|------------------------|-------------------|
| **触发** | 达到阈值时自动触发 | 由你实现阈值检查 |
| **摘要** | SDK 处理摘要请求 | 由你发起显式 API 调用 |
| **历史管理** | SDK 替换消息 | 由你手动替换列表 |
| **适用场景** | 带工具的智能体工作流 | 用户主导的对话 |

**生产环境注意事项**：

1. **调整阈值**：在真实应用中使用更大的阈值
2. **自定义摘要提示词**：针对你的对话类型做定制（头脑风暴、技术支持还是辅导）
3. **向用户显示指示信息**：显示类似“正在总结对话……”的消息，让用户理解这次暂停
4. **保留关键上下文**：确保摘要提示词能捕捉用户关心的领域特定信息

这种模式让你完全掌控压缩发生的时机和方式，因此非常适合无法使用 SDK 自动工具运行器压缩的对话类应用。

## 局限与注意事项

尽管自动上下文压缩功能强大，但有一些重要的局限需要了解：

### 服务端采样循环

**当前局限**：压缩在服务端采样循环下无法达到最佳效果，例如服务端网络搜索工具。

**原因**：缓存词元会在采样循环中累积，可能基于缓存内容而非真实对话历史过早触发压缩。

该功能最适合以下场景：
- ✅ 客户端工具（如本 cookbook 中的客服 API）
- ✅ 使用常规工具使用的标准智能体工作流
- ✅ 文件操作、数据库查询、API 调用
- ❌ 服务端扩展思考
- ❌ 服务端网络搜索工具

### 信息丢失

**权衡**：摘要必然会丢失一些信息。尽管 Claude 擅长识别要点，仍有一些细节会被压缩或省略。

**在工单处理中**：
- ✅ **保留**：工单 ID、类别、优先级、团队、结果、进度状态
- ❌ **丢失**：完整的知识库文章正文、完整的响应草稿正文、详细的分类推理过程

这通常可以接受：你并不需要永久保留每一篇知识库文章和完整的响应正文，只需要完成情况记录。

**缓解措施**：
- 使用自定义摘要提示词来保留关键信息
- 对需要大量历史上下文的任务设置更高的阈值
- 把任务设计成模块化的（每个阶段基于摘要构建，而不是基于原始细节）

### 何时不应使用压缩

以下场景应避免压缩：

1. **短任务**：如果任务在 50k-100k 词元内就能完成，压缩会带来不必要的开销
2. **需要完整审计痕迹的任务**：有些任务必须访问此前的全部细节
3. **服务端采样工作流**：如上所述，等待这一局限被解决
4. **高度迭代打磨**：每一步都关键依赖此前所有步骤的精确细节的任务

### 何时应当使用压缩

压缩最适合：

1. **顺序处理**：如我们的工单工作流——依次处理多个条目
2. **多阶段工作流**：每个阶段在继续之前都可以先总结进度
3. **迭代式数据处理**：按数据块或逐个实体处理大型数据集
4. **长时间分析会话**：跨多个实体分析数据
5. **批次操作**：处理数百个彼此独立的条目

**工单处理是完美的用例**，因为：
- 每张工单的工作流大体上彼此独立
- 你需要的是完成情况摘要，而不是完整的工具结果
- 存在自然的压缩点（完成若干张工单之后）
- 工作流是迭代且顺序的

## 总结

自动上下文压缩是一项强大的功能，能让长时间运行的智能体工作流突破常见的上下文限制。在本 cookbook 中，我们通过一个客服工单处理工作流探讨了压缩。

### 下一步

尝试在你自己的工作流中实现压缩：
1. 找出自然的压缩点（处理完每个条目之后、完成每个阶段之后等）
2. 如果每个条目的边界清晰，先用一个较激进的阈值（5k-10k）
3. 使用自定义摘要提示词来保留关键信息
4. 监控压缩何时触发，并确认质量得以保持
5. 根据自身具体需求调整阈值

关于有效上下文管理的更多内容，参见[面向 AI 智能体的有效上下文工程](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)。